# Day 25 — Binomial & Poisson
### Statistics & Probability · Module 2 · Topic 2.5

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours · 90 min concepts + 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | What makes a situation **binomial** — four conditions | 20 min |
| 2 | **Building** the binomial formula from 8 coin flips | 25 min |
| 3 | Using it — pmf, cdf, mean and spread | 20 min |
| 4 | **Poisson** — when n is huge and p is tiny | 25 min |
| 5 | Choosing between them, and where both lead | 10 min |
| 6 | **Practice notebook (separate file)** | 30 min |

---

## The step we take today

Days 23 and 24 were about **measuring**: how tall, how much, how long. The answer could be
any number, including fractions. Those are **continuous**.

Today we **count**: how many of 20 items are defective, how many visitors clicked, how many
calls arrived this hour. The answer is always a whole number — 0, 1, 2, 3 — and never 2.7.
Those are **discrete**.

That difference matters more than it sounds. For continuous data, the probability of a height
being *exactly* 171.30000… cm is zero — we could only ever ask about a range. For discrete
data, "what is the probability of exactly 3?" has a real, useful answer. Two distributions
cover almost every counting problem you will meet, and one question tells you which to use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from math import comb, factorial, exp
import warnings
warnings.simplefilter("ignore")

plt.rcParams["figure.figsize"] = (8, 3.4)
print("numpy", np.__version__, "| pandas", pd.__version__)

---
# 1. What makes a situation binomial

All four of these must hold:

| # | Condition | What it means |
|---|---|---|
| 1 | A **fixed** number of trials | You decide `n` before you start |
| 2 | Each trial has **two** outcomes | success / failure, defective / not |
| 3 | The **same** probability every time | `p` does not drift between trials |
| 4 | Trials are **independent** | one result tells you nothing about the next |

Miss any one of them and the formula still returns a number — a confidently wrong one, with
no error message. So walk the four in order and find the first that fails.

### Does it qualify?

| Situation | Verdict | Why |
|---|---|---|
| Flip a coin 10 times, count heads | ✅ binomial | n=10 · H/T · p=0.5 always · flips independent |
| Inspect 20 items from a large batch | ✅ binomial | n=20 · defective or not · p=5% each |
| 500 visitors see an ad, count clicks | ✅ binomial | n=500 · click or not · same p · independent |
| Flip **until** you get a head | ❌ | n is not fixed — **condition 1 fails** (this is *geometric*) |
| Draw 5 cards from one deck, count aces | ❌ | p goes 4/52, then 3/51… — **condition 3 fails** (*hypergeometric*) |
| Did 3 family members catch flu? | ❌ | they infect each other — **condition 4 fails** |

The card example is the subtle one. *With* replacement it would be binomial; it is the
not-replacing that breaks `p`.

---
# 2. Building the binomial from scratch

Nobody should memorise this formula. Let us derive it.

## 2.1 Start small — flip a coin three times

There are 8 equally likely sequences. Let us list every one and group them by how many heads
they contain.

In [ ]:
from itertools import product

outcomes = ["".join(t) for t in product("HT", repeat=3)]
print("all", len(outcomes), "outcomes:", outcomes)
print()

for k in range(4):
    matching = [o for o in outcomes if o.count("H") == k]
    print(f"{k} head(s): {len(matching)} way(s)  ->  {matching}")

There is only **one** way to get zero heads (TTT) but **three** ways to get exactly one — the
head can sit in any of the three positions.

The counts are **1, 3, 3, 1**. Every sequence is equally likely, so the probability of a given
number of heads is just *how many sequences give that number, divided by 8*.

In [ ]:
counts = np.array([1, 3, 3, 1])
probs  = counts / 8
print("by hand :", probs)
print("scipy   :", stats.binom.pmf([0, 1, 2, 3], n=3, p=0.5))
print("they match:", np.allclose(probs, stats.binom.pmf([0,1,2,3], 3, 0.5)))
print("and they sum to", probs.sum())

In [ ]:
ks = np.arange(4)
pr = stats.binom.pmf(ks, 3, 0.5)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar(ks, pr, color="#3E7CB1", width=0.6)
for k, v in zip(ks, pr):
    ax.text(k, v + 0.012, f"{v:.3f}", ha="center", fontweight="bold")
ax.set_xticks(ks)
ax.set_xlabel("Number of heads"); ax.set_ylabel("Probability")
ax.set_title("Binomial(n=3, p=0.5)"); ax.set_ylim(0, 0.46)
plt.show()

Each **bar** is a separate possible count. There is nothing between them — you cannot get 1.5
heads, so there is no bar there. The heights add to exactly 1.000, because one of those four
things must happen.

That is what *discrete distribution* means in practice: a finite list of outcomes, each with
its own probability.

## 2.2 Piece one — the probability of ONE path

Take any particular sequence with 3 successes out of 5, say **S F S S F**. The trials are
independent, so we multiply along the path:

```
p  ×  (1-p)  ×  p  ×  p  ×  (1-p)   =   p³ (1-p)²
```

And here is the hinge of the whole derivation: **any other path** with 3 successes and 2
failures gives exactly the same product, because multiplication does not care about order.
So every such path has probability `p³(1-p)²`. We only need to know how many there are.

## 2.3 Piece two — how many paths

"How many ways to choose k positions out of n" — that is exactly `C(n, k)`.

In [ ]:
print("comb(3,0..3):", [comb(3, k) for k in range(4)], " <- our 1, 3, 3, 1")
print()
print("C(n, k) = n! / (k! (n-k)!)")
print("C(5, 3) =", factorial(5), "/ (", factorial(3), "*", factorial(2), ") =", comb(5, 3))

## 2.4 Put the two pieces together

$$P(k) \;=\; \underbrace{C(n,k)}_{\text{how many paths}} \times \underbrace{p^{k}}_{k \text{ successes}} \times \underbrace{(1-p)^{\,n-k}}_{\text{the rest are failures}}$$

Let us do one entirely by hand and check it against scipy.

In [ ]:
n, k, p = 10, 7, 0.5

ways     = comb(n, k)
successes = p ** k
failures  = (1 - p) ** (n - k)
by_hand   = ways * successes * failures

print(f"C({n},{k})          = {ways}")
print(f"p^k      = {p}^{k}  = {successes}")
print(f"(1-p)^(n-k) = {1-p}^{n-k} = {failures}")
print("-" * 40)
print(f"product            = {by_hand:.6f}")
print(f"scipy              = {stats.binom.pmf(k, n, p):.6f}")

About an **11.7%** chance of exactly 7 heads in 10 flips.

Sanity check first, arithmetic second: 7 out of 10 is above the expected 5, so the answer
should be smallish but not tiny. 11.7% feels right. Had you got 0.9 or 0.001, something
would be wrong.

## 2.5 What `n` and `p` do to the shape

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 5))
for ax, (n_, p_) in zip(axes.ravel(),
                        [(10, .5), (10, .2), (10, .8), (5, .5), (20, .5), (50, .5)]):
    ks = np.arange(n_ + 1)
    ax.bar(ks, stats.binom.pmf(ks, n_, p_), color="#3E7CB1")
    ax.axvline(n_ * p_, color="#E8674C", ls="--", lw=2)
    ax.set_title(f"n={n_}, p={p_}   mean={n_*p_:g}", fontsize=10)
fig.suptitle("p moves the peak  ·  n stretches and smooths it")
plt.tight_layout(); plt.show()

**p** moves the peak — it sits at `n × p`. **n** widens the distribution and smooths it.

Look hard at the `n=50` panel. That is starting to look like yesterday's bell curve. Hold
that thought; we come back to it at the end of the session.

---
# 3. Using the binomial

Our running example: **20 items inspected, 5% defect rate.**

Three different questions, the same set of bars.

In [ ]:
n, p = 20, 0.05

exactly_1 = stats.binom.pmf(1, n, p)
at_most_1 = stats.binom.cdf(1, n, p)
at_least_2 = 1 - stats.binom.cdf(1, n, p)

print(f"EXACTLY 1  pmf(1)       = {exactly_1:.4f}")
print(f"AT MOST 1  cdf(1)       = {at_most_1:.4f}")
print(f"AT LEAST 2 1 - cdf(1)   = {at_least_2:.4f}")
print()
print("at most 1 + at least 2 =", round(at_most_1 + at_least_2, 10), "<- must be 1")

In [ ]:
ks = np.arange(0, 9)
pr = stats.binom.pmf(ks, n, p)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, mask, title, col in zip(
        axes,
        [ks == 1, ks <= 1, ks >= 2],
        ["EXACTLY 1\npmf(1) = 0.377", "AT MOST 1\ncdf(1) = 0.736", "AT LEAST 2\n1-cdf(1) = 0.264"],
        ["#FFC145", "#2FA98C", "#E8674C"]):
    ax.bar(ks, pr, color="#D8E1EB")
    ax.bar(ks[mask], pr[mask], color=col)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(ks); ax.set_xlabel("defective items")
axes[0].set_ylabel("probability")
plt.tight_layout(); plt.show()

## ⚠️ 3.1 The off-by-one that gets everyone once

**"At least 2" is NOT `1 - cdf(2)`.**

`cdf(2)` already *includes* the bar at 2. Subtracting it removes the very outcome you wanted
to keep.

In [ ]:
wrong = 1 - stats.binom.cdf(2, n, p)
right = 1 - stats.binom.cdf(1, n, p)

print(f"1 - cdf(2) = {wrong:.4f}   <- WRONG, this is 'at least 3'")
print(f"1 - cdf(1) = {right:.4f}   <- right, 'at least 2'")
print(f"off by a factor of {right / wrong:.1f}")

Both look like plausible probabilities. Nothing errors, nothing warns you.

**The rule that always works:**

```
P(at least k)  =  1 - cdf(k - 1)
```

Say it out loud: *"at least 2 means NOT 0 and NOT 1"*, so subtract everything up to and
including 1. When in doubt, draw the bars and shade the ones you want.

This is the discrete-distribution cousin of Day 16's `loc` vs `iloc`: an off-by-one that
produces a confident, wrong number.

## 3.2 Mean and spread

In [ ]:
mean = n * p
sd   = np.sqrt(n * p * (1 - p))

print(f"mean = n*p            = {mean:.4f}")
print(f"sd   = sqrt(n*p*(1-p)) = {sd:.4f}")
print()
m_, v_ = stats.binom.stats(n, p, moments="mv")
print(f"scipy: mean {float(m_):.4f}, variance {float(v_):.4f} (sd = {float(v_)**0.5:.4f})")

# a sanity check you can do in your head
observed = 8
print(f"\nA batch of 20 comes back with {observed} defects.")
print(f"That is {(observed - mean) / sd:.1f} standard deviations above the expected {mean:.0f}.")
print("Day 24's ruler, applied to counts - you do not need a formal test to be suspicious.")

In [ ]:
# the spread is largest at p = 0.5 and shrinks towards 0 and 1
ps = np.linspace(0.01, 0.99, 99)
sds = np.sqrt(20 * ps * (1 - ps))

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot(ps, sds, color="#2FA98C", lw=2.5)
ax.axvline(0.5, color="#E8674C", ls="--")
ax.set_xlabel("p"); ax.set_ylabel("standard deviation")
ax.set_title("n = 20: spread peaks at p = 0.5")
plt.show()

print("An event that almost never happens, or almost always happens, is predictable.")

---
# 4. Poisson — when there is no `n`

## 4.1 A question the binomial cannot answer

> *"A support desk averages 3 calls an hour. What is the chance of 5 or more this hour?"*

Try to force that into a binomial and the first question is: **what is n?** Every second is a
chance for a call — so 3,600? Every millisecond — so 3,600,000? There is no natural answer.
And `p` would be whatever makes the numbers work out, which is not a definition.

Same problem for:

- typos per page of a book
- accidents at a junction per month
- customers entering a shop per 10 minutes
- server errors per day

What they share: events happen at some average **rate** over a continuous interval of time or
space. There are no countable trials — just an interval, and events landing in it.

## 4.2 One number: λ (lambda)

**λ = the average count per interval.** That single number fixes the whole distribution —
there is no second parameter to find.

$$P(k) \;=\; \frac{e^{-\lambda}\,\lambda^{k}}{k!}$$

Notice what is *not* in it: no `n`, no `p`.

> ⚠️ **λ must match your interval.** 3 per hour means λ=3 for an hour, λ=1.5 for 30 minutes,
> λ=72 for a day. Getting this wrong is the most common Poisson mistake in production code.

In [ ]:
lam = 3

# by hand, then scipy
by_hand = exp(-lam) * lam ** 2 / factorial(2)
print(f"P(exactly 2) by hand = {by_hand:.6f}")
print(f"P(exactly 2) scipy   = {stats.poisson.pmf(2, lam):.6f}")
print()

# scaling lambda to the interval
for label, hours in [("20 minutes", 1/3), ("1 hour", 1), ("8-hour shift", 8), ("a day", 24)]:
    print(f"{label:>14}:  lambda = {3 * hours:6.2f}")

### The three Poisson conditions

| Condition | Meaning |
|---|---|
| Events are **independent** | one call does not cause another |
| The rate is **constant** | 3 an hour all through the interval |
| Two cannot land at the **same instant** | events are separable |

The constant-rate condition fails more often than people expect: a call centre at 9am and at
3am do not share a λ.

## 4.3 The call-centre example, worked through

In [ ]:
lam = 3
ks = np.arange(0, 11)
pr = stats.poisson.pmf(ks, lam)

questions = {
    "no calls at all   P(0)":      stats.poisson.pmf(0, lam),
    "exactly 3         P(3)":      stats.poisson.pmf(3, lam),
    "at most 2         cdf(2)":    stats.poisson.cdf(2, lam),
    "5 or more     1 - cdf(4)":    1 - stats.poisson.cdf(4, lam),
}
for q, v in questions.items():
    print(f"{q:<28} = {v:.4f}")

print()
print(f"mean = {lam},  variance = {lam},  sd = {np.sqrt(lam):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
cols = ["#2FA98C" if k < 5 else "#E8674C" for k in ks]
ax.bar(ks, pr, color=cols)
for k in range(6):
    ax.text(k, pr[k] + 0.006, f"{pr[k]:.3f}", ha="center", fontsize=9)
ax.plot([lam, lam], [0, 0.196], color="#152238", ls="--", lw=1.8)
ax.text(lam + 0.18, 0.199, "mean = $\\lambda$ = 3")
ax.set_xticks(ks); ax.set_xlabel("Calls in one hour"); ax.set_ylabel("Probability")
ax.set_title("Poisson($\\lambda$=3): red bars are '5 or more' = 0.185")
plt.show()

Note that **2 and 3 are equally likely** — that happens whenever λ is a whole number.

The staffing payoff: **18.5% of hours bring 5 or more calls.** If you staff for the average of
3, you are overwhelmed roughly one hour in five. An average alone could never have told you
that; the distribution does.

## 4.4 How λ changes the shape

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 2.9))
for ax, lm in zip(axes, [0.5, 2, 5, 15]):
    hi = max(8, int(lm + 4 * np.sqrt(lm)) + 1)
    kk = np.arange(hi)
    ax.bar(kk, stats.poisson.pmf(kk, lm), color="#2FA98C")
    ax.axvline(lm, color="#E8674C", ls="--", lw=2)
    ax.set_title(f"$\\lambda$ = {lm}")
plt.tight_layout(); plt.show()

Counts cannot go below zero. With a small λ the distribution piles up against that floor and
has a long right tail — strongly skewed. With a large λ the floor is far away and the shape is
nearly symmetric.

## ⭐ 4.5 The Poisson's strangest property: mean = variance = λ

One number fixes both the centre **and** the spread. No other common distribution does this,
and it gives you a free diagnostic.

In [ ]:
rng = np.random.default_rng(2505)

# data that really is Poisson
clean = rng.poisson(4, size=5000)

# data that is NOT - events arrive in bursts (a link gets shared)
bursty = np.concatenate([rng.poisson(1, 4500), rng.poisson(30, 500)])

for name, d in [("clean Poisson", clean), ("bursty traffic", bursty)]:
    ratio = d.var() / d.mean()
    verdict = "Poisson is plausible" if ratio < 1.5 else "OVERDISPERSED - not Poisson"
    print(f"{name:<16} mean {d.mean():6.2f}  var {d.var():7.2f}  ratio {ratio:5.2f}   {verdict}")

If the variance is much bigger than the mean, the data is **overdispersed** — something is
clustering the events, which breaks the independence condition.

Website visits per minute usually fail this badly: visits arrive in bursts when a link is
shared. Earthquakes fail it too, because aftershocks cluster. The mean-variance ratio catches
both without any formal test.

---
# 5. Choosing between them, and where both lead

## 5.1 One question decides it

> ### "Can I count the number of trials?"

| Answer | Use | Examples |
|---|---|---|
| **Yes, there is a clear n** | Binomial | 20 items inspected · 10 coin flips · 500 visitors · a class of 60 |
| **No, just a rate** | Poisson | calls per hour · typos per page · accidents per month |

A crude but reliable tell: if the question says **"out of"**, it is binomial. If it says
**"per"**, it is Poisson.

## 5.2 They are not really two different things

In [ ]:
ks = np.arange(0, 11)
print("Binomial(n, p) vs Poisson(3), all with n x p = 3")
print(f"{'n':>7} {'p':>8} {'binom P(2)':>12} {'pois P(2)':>11} {'difference':>12}")
for n_, p_ in [(10, .3), (50, .06), (200, .015), (2000, .0015), (10000, .0003)]:
    b = stats.binom.pmf(2, n_, p_)
    po = stats.poisson.pmf(2, 3)
    print(f"{n_:>7} {p_:>8} {b:>12.6f} {po:>11.6f} {abs(b-po):>12.6f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
for ax, (n_, p_) in zip(axes, [(10, .3), (50, .06), (200, .015), (2000, .0015)]):
    ax.bar(ks - .2, stats.binom.pmf(ks, n_, p_), width=.4, color="#3E7CB1", label="Binomial")
    ax.bar(ks + .2, stats.poisson.pmf(ks, 3), width=.4, color="#E8674C", label="Poisson(3)")
    ax.set_title(f"n={n_}, p={p_:g}", fontsize=10)
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle("Same mean throughout - as n grows and p shrinks, the two become identical")
plt.tight_layout(); plt.show()

Poisson is what the binomial **becomes** when `n` grows and `p` shrinks with `n × p` held
fixed. That is why the old rule of thumb exists — *if n > 50 and p < 0.05 you may use Poisson
with λ = n×p*. In practice you will just use the binomial, because computers do not care; the
value of the rule is that it explains why Poisson works for rare events at all.

## 5.3 And both become yesterday's bell curve

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 2.9))
for ax, lm in zip(axes, [1, 5, 20, 100]):
    lo, hi = max(0, int(lm - 4.5*np.sqrt(lm))), int(lm + 4.5*np.sqrt(lm)) + 2
    kk = np.arange(lo, hi)
    ax.bar(kk, stats.poisson.pmf(kk, lm), color="#2FA98C")
    xs = np.linspace(kk[0], kk[-1], 300)
    ax.plot(xs, stats.norm.pdf(xs, lm, np.sqrt(lm)), color="#152238", lw=2)
    ax.set_title(f"$\\lambda$={lm}   skew={1/np.sqrt(lm):.2f}", fontsize=10)
plt.tight_layout(); plt.show()

print("Poisson skew is exactly 1/sqrt(lambda):")
for lm in [1, 5, 20, 100]:
    print(f"  lambda={lm:>4}   theory {1/np.sqrt(lm):.3f}   scipy {stats.poisson.stats(lm, moments='s'):.3f}")

That is **twice in one session** the normal curve has turned up uninvited — once in the `n=50`
binomial panel, once here. Day 27 explains why it keeps happening.

## 5.4 Reference table

| | Binomial | Poisson |
|---|---|---|
| Question it answers | how many successes **out of** n? | how many events **per** interval? |
| Parameters | `n` and `p` | `λ` only |
| Possible values | 0 to n | 0 upward, no limit |
| Mean | `n × p` | `λ` |
| Variance | `n × p × (1-p)` | `λ` (same as the mean) |
| Exactly k | `binom.pmf(k, n, p)` | `poisson.pmf(k, lam)` |
| At most k | `binom.cdf(k, n, p)` | `poisson.cdf(k, lam)` |
| At least k | `1 - binom.cdf(k-1, n, p)` | `1 - poisson.cdf(k-1, lam)` |

Both are discrete, so both use `pmf` for *exactly* and `cdf` for *at most*. Neither has a
`pdf` — that is for continuous distributions like yesterday's normal.

---
# 6. Mini build — a staffing decision

The question an average cannot answer.

In [ ]:
from scipy import stats

#  A support desk averages 3 calls an hour.
#  One agent can handle 4 calls an hour.
#  QUESTION: how often are we overwhelmed?

lam = 3

#  --- 1. the risk with one agent
overwhelmed = 1 - stats.poisson.cdf(4, lam)
print(f"P(5 or more calls) = {overwhelmed:.4f}  -> about one hour in {1/overwhelmed:.0f}")

#  --- 2. what would two agents give us?
two = 1 - stats.poisson.cdf(8, lam)
print(f"P(9 or more calls) = {two:.4f}  -> about one hour in {1/two:.0f}")

#  --- 3. express it in a unit people feel
hours = 8 * 22                      # a working month
print()
for agents in (1, 2, 3):
    cap = 4 * agents
    risk = 1 - stats.poisson.cdf(cap, lam)
    print(f"{agents} agent(s): overwhelmed {risk*hours:6.1f} hours/month")

Six things worth noticing in that cell:

1. **λ matched the interval** — 3 per hour, asked per hour.
2. **`cdf(4)`, not `cdf(5)`** — "5 or more" is `1 - cdf(4)`.
3. The result is a **probability, not a count** — 0.185 *of hours*, not 0.185 calls.
4. We translated it to a unit people feel — "hours per month" lands where "0.185" does not.
5. We **compared the options**; a decision needs alternatives.
6. **The average alone was useless.** Average load 3, capacity 4, so an average-based answer
   says *"we are fine"* — and is wrong one hour in five.

---

# Debug corner — eight counting mistakes

| You did | What goes wrong | Fix |
|---|---|---|
| used `1 - cdf(k)` for "at least k" | off by one — excludes k itself | `1 - cdf(k-1)` |
| binomial on cards without replacement | p changes every draw | hypergeometric distribution |
| binomial when n is not fixed | "flip until a head" has no n | geometric distribution |
| λ for the wrong interval | 3/hour is not 3 per 20 minutes | scale λ to your window |
| Poisson when events cluster | variance far exceeds the mean | check `var/mean` before modelling |
| `pdf` on a discrete distribution | there is no such method | `pmf` for counts, `pdf` for continuous |
| forgot `p` in `binom.pmf(k, n)` | TypeError | always pass n **and** p |
| asked `P(exactly 2.5)` | counts are whole numbers | the question is malformed |

Row 1 will cost you marks. Row 4 will cost you a production incident.

In [ ]:
# proof that two of those really do fail
try:
    stats.binom.pmf(2, 10)
except TypeError as e:
    print("binom.pmf(2, 10)  ->  TypeError:", str(e)[:60], "...")

print("poisson has a pmf? ", hasattr(stats.poisson, "pmf"))
print("poisson has a pdf? ", hasattr(stats.poisson, "pdf"))
print("P(exactly 2.5) =", stats.poisson.pmf(2.5, 3), " <- zero, the question was malformed")

---
# Recap — today in twelve lines

1. Days 23–24 measured quantities. Today we **count** events.
2. Counts are whole numbers, so we get **bars**, not a curve.
3. Binomial needs four things: fixed n, two outcomes, same p, independence.
4. Its formula is `C(n,k) × p^k × (1-p)^(n-k)` — paths × successes × failures.
5. `C(n,k)` is just *how many orders give k successes*. For n=3: 1, 3, 3, 1.
6. Binomial mean = `np`, sd = `sqrt(np(1-p))`.
7. ⚠️ "At least k" is `1 - cdf(k-1)`. Using `cdf(k)` is the classic error.
8. Poisson is for a **rate** with no countable n: calls per hour, typos per page.
9. It needs one number, λ — and **λ must match your interval**.
10. Poisson **mean = variance = λ**. If var ≫ mean, events are clustering.
11. The deciding question: *"out of"* → binomial, *"per"* → Poisson.
12. Binomial → Poisson as n grows; both → Normal. Day 27 says why.

### Homework
- Write out all 16 outcomes of 4 coin flips and check they give `C(4,k)`.
- Find three situations at your own workplace or studies — one binomial, one Poisson — and say
  which condition decided it.
- Redo the staffing build with λ = 8. How many agents do you need to be overwhelmed less than
  one hour a month?

### Next class — Topic 2.6: The Uniform Distribution
When every outcome is equally likely. The simplest distribution of all, and the one every
random number generator is built on.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*